# Camada Bronze — Ingestão de Dados Brutos


## 0. Parâmetros do Notebook

In [0]:
# Parâmetros configuráveis via Databricks Widgets
dbutils.widgets.text("data_inicio", "01-01-2021", "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim",    "12-31-2021", "Data Fim   (MM-DD-AAAA)")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim    = dbutils.widgets.get("data_fim")

print(f"Período de cotação: {data_inicio} → {data_fim}")

Período de cotação: 01-01-2021 → 12-31-2021


## 1. Imports e Configurações

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType, TimestampType
)
import requests
from datetime import datetime

# Caminho base onde os CSVs foram carregados como Volume no Databricks
VOLUME_PATH = "/Volumes/workspace/default/olist_data"

# Catalog e schema da camada Bronze
CATALOGO  = "workspace"
DB_BRONZE = "bronze"

print("Imports realizados com sucesso.")

Imports realizados com sucesso.


## 2. Criação do Banco de Dados Bronze

In [0]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOGO}.{DB_BRONZE}")
print(f"Banco de dados '{CATALOGO}.{DB_BRONZE}' criado (ou já existia).")

Banco de dados 'workspace.bronze' criado (ou já existia).


## 3. Função Auxiliar de Ingestão CSV → Delta

In [0]:
def ingerir_csv(nome_arquivo: str, nome_tabela: str):
    tabela_completa = f"{CATALOGO}.{DB_BRONZE}.{nome_tabela}"
    caminho = f"{VOLUME_PATH}/{nome_arquivo}"

    try:
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("encoding", "UTF-8")
            .csv(caminho)
        )

        if df.count() == 0:
            raise ValueError(f"O arquivo '{nome_arquivo}' está vazio ou não pôde ser lido.")

        df = df.withColumn("timestamp_ingestion", F.current_timestamp())

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(tabela_completa)
        )

        count = spark.table(tabela_completa).count()
        print(f"✅ {tabela_completa:<60} → {count:>10,} registros")

    except Exception as e:
        print(f"❌ Erro ao processar '{nome_tabela}': {str(e)}")

## 4. Ingestão dos Arquivos CSV

In [0]:
mapeamento = {
    "olist_customers_dataset.csv"            : "tb_customers",
    "olist_geolocation_dataset.csv"          : "tb_geolocalizacao",
    "olist_order_items_dataset.csv"          : "tb_order_items",
    "olist_order_payments_dataset.csv"       : "tb_order_payments",
    "olist_order_reviews_dataset.csv"        : "tb_order_reviews",
    "olist_orders_dataset.csv"               : "tb_orders",
    "olist_products_dataset.csv"             : "tb_products",
    "olist_sellers_dataset.csv"              : "tb_sellers",
    "product_category_name_translation.csv"  : "tb_product_category_name_translation",
}

print("Iniciando ingestão dos arquivos CSV...\n")
for arquivo, tabela in mapeamento.items():
    ingerir_csv(arquivo, tabela)

print("\nIngestão dos CSVs concluída!")

Iniciando ingestão dos arquivos CSV...

✅ workspace.bronze.tb_customers                                →     99,441 registros
✅ workspace.bronze.tb_geolocalizacao                           →  1,000,163 registros
✅ workspace.bronze.tb_order_items                              →    112,650 registros
✅ workspace.bronze.tb_order_payments                           →    103,886 registros
✅ workspace.bronze.tb_order_reviews                            →    104,162 registros
✅ workspace.bronze.tb_orders                                   →     99,441 registros
✅ workspace.bronze.tb_products                                 →     32,951 registros
✅ workspace.bronze.tb_sellers                                  →      3,095 registros
✅ workspace.bronze.tb_product_category_name_translation        →         71 registros

Ingestão dos CSVs concluída!


## 5. Ingestão da Cotação do Dólar via API do Banco Central

In [0]:
def buscar_cotacao_dolar(data_inicio: str, data_fim: str) -> list:
    url = (
        "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata"
        "/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
        f"?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
        "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
    )

    print(f"Consultando API: {url[:80]}...")
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    dados = response.json().get("value", [])
    print(f"Registros retornados pela API: {len(dados)}")
    return dados

In [0]:
dados_cotacao = buscar_cotacao_dolar(data_inicio, data_fim)

if dados_cotacao:
    schema_cotacao = StructType([
        StructField("dataHoraCotacao", StringType(), True),
        StructField("cotacaoCompra",   DoubleType(), True),
    ])

    df_cotacao = (
        spark.createDataFrame(dados_cotacao, schema=schema_cotacao)
        .withColumn("timestamp_ingestion", F.current_timestamp())
    )

    tabela_cotacao = f"{CATALOGO}.{DB_BRONZE}.tb_cotacao_dolar"

    (
        df_cotacao.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_cotacao)
    )

    count = spark.table(tabela_cotacao).count()
    print(f"✅ {tabela_cotacao} → {count:,} registros")
else:
    print("⚠️  API não retornou dados para o período informado.")

Consultando API: https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriod...
Registros retornados pela API: 251
✅ workspace.bronze.tb_cotacao_dolar → 251 registros


## 6. Verificação Final

In [0]:
print("=" * 60)
print("TABELAS CRIADAS NA CAMADA BRONZE")
print("=" * 60)
spark.sql(f"SHOW TABLES IN {CATALOGO}.{DB_BRONZE}").show(truncate=False)

TABELAS CRIADAS NA CAMADA BRONZE
+--------+------------------------------------+-----------+
|database|tableName                           |isTemporary|
+--------+------------------------------------+-----------+
|bronze  |tb_cotacao_dolar                    |false      |
|bronze  |tb_customers                        |false      |
|bronze  |tb_geolocalizacao                   |false      |
|bronze  |tb_order_items                      |false      |
|bronze  |tb_order_payments                   |false      |
|bronze  |tb_order_reviews                    |false      |
|bronze  |tb_orders                           |false      |
|bronze  |tb_product_category_name_translation|false      |
|bronze  |tb_products                         |false      |
|bronze  |tb_sellers                          |false      |
+--------+------------------------------------+-----------+

